# Stg Layer — videos (every read, insert-only)

Read **videos** from Bronze (`${bronze}.videos_raw`, append-only copy of Lakebase Postgres) → Silver `${silver}.stg_videos`.

**Grain (since 2026-11-02):** 1 row per `video_id` × **`snapshot_ts`** (= `COALESCE(updated_at, created_at)`, when the app read the values from YouTube). Every read is kept, so the 07 / 09 / 21h stats refreshes no longer overwrite each other (before: 1 row per video_id per day, the latest read of the day won). Nothing is updated: insert-only MERGE on (`video_id`, `snapshot_ts`). `snapshot_date` = the Vietnam day of `snapshot_ts`.

**Current values:** view `${silver}.stg_videos_latest` (latest read per `video_id`, same columns).

**Flow:** `%run ./_history` → `history_load`: watermark `MAX(snapshot_ts)`, new Bronze rows, one row per grain, insert-only MERGE, refresh the `_latest` view, grain check. The first run after 2026-11-02 renames the old table to `stg_videos_bak_20261102` and rebuilds from Bronze + that table.

In [ ]:
%run ./_history

In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

SOURCE_TABLE = f"{BRONZE}.videos_raw"  # Bronze: append-only copy of every Postgres read
DELTA_TABLE = f"{SILVER}.stg_videos"
print(f"Bronze source: {SOURCE_TABLE} · Silver target: {DELTA_TABLE} (+ view {DELTA_TABLE}_latest)")

In [ ]:
DDL = """
    video_id       STRING,
    snapshot_ts    TIMESTAMP,
    snapshot_date  DATE,
    channel_id     STRING,
    channel_title  STRING,
    title          STRING,
    description    STRING,
    published_at   TIMESTAMP,
    category_id    STRING,
    tags           ARRAY<STRING>,
    duration       STRING,
    definition     STRING,
    has_caption    BOOLEAN,
    view_count     BIGINT,
    like_count     BIGINT,
    comment_count  BIGINT,
    thumbnail_url  STRING,
    video_url      STRING,
    created_at     TIMESTAMP,
    updated_at     TIMESTAMP,
    source         STRING,
    etl_date       TIMESTAMP,
    op_type        STRING
"""
added = history_load(SOURCE_TABLE, DELTA_TABLE, "video_id", DDL)

In [ ]:
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT video_id) AS ids, COUNT(DISTINCT snapshot_date) AS days, MAX(snapshot_ts) AS last_read
    FROM {DELTA_TABLE}
""").collect()[0]
print(f"📊 {DELTA_TABLE}: {s['n']} reads of {s['ids']} video_ids over {s['days']} day(s), last read {s['last_read']}, added this run: {added}")
display(spark.sql(f"SELECT snapshot_date, COUNT(*) AS reads, COUNT(DISTINCT video_id) AS ids, COUNT(DISTINCT snapshot_ts) AS read_times FROM {DELTA_TABLE} GROUP BY 1 ORDER BY 1 DESC LIMIT 14"))
print("\n--- stg_videos ETL complete ---")